## 1. Prepare data

This script prepares the variables as .csv files and converts them into the .zarr files needed for the model. This takes some times.

Due to the set-up, we need to create a separate `.zarr` file for every set of covariates we want to run (each using its own function from fore). The models are the following:

- **M1**: only initial recruitment (years 2 - 10)
- **M2**: only climate (binned in 25 year climatologies along the trajectory)
- **M3**: only soil properties
- **M4**: all of the above

*We leave out previous state because in the abense of climate change that will just be a proxy for everything else and will not offer any insights*

In [1]:
!pip install zarr
!pip install matplotlib
!pip install pandas
!pip install scikit-learn

import zarr

from  fore.data import ReadMLData_m1
from  fore.data import ReadMLData_m2
from  fore.data import ReadMLData_m3
from  fore.data import ReadMLData_m4
from  fore.data import ReadMLData_m5
from  fore.data import ReadMLData_m6

import numpy as np
import matplotlib.pyplot as plt

  Using cached matplotlib-3.9.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached contourpy-1.2.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (162 kB)
  Using cached kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylinux2010_x86_64.whl.metadata (6.4 kB)
  Using cached pillow-10.4.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (9.2 kB)
Using cached matplotlib-3.9.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.3 MB)
Using cached contourpy-1.2.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (305 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.53.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.6 MB)
Using cached kiwisolver-1.4.5-cp310-cp310-manylinux_2_12_x86_64.manylin

### Generate data for model 1 (only recruitment)

In [2]:
for fname in ["ssp585_2015_2040", "ssp126_2015_2040", "picontrol_2015_2040",
              "ssp585_2075_2100","ssp126_2075_2100", "picontrol_2075_2100"]: #add path of data

    ds_path = f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/data_{fname}.csv"

    data = ReadMLData_m1(ds_path)
    x_t, x_s, y_start, y_end, y_block, y, lat_lon_pid = data.generate_ml_data(block_size=25)
    index = ~np.isnan(y_end.sum(axis=1))
    print(sum(index))
    print(y_end.shape[0])

    x_t = x_t[index]
    x_s = x_s[index]
    y_start = y_start[index]
    y_end = y_end[index]
    y_block = y_block[index] 
    y = y[index]
    lat_lon_pid = lat_lon_pid[index]

    
    with zarr.open(f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/{fname}_m1.zarr", "w") as f: #correct path
    
        f["x_t"] = x_t
        f["x_s"] = x_s
        f["y_start"] = y_start
        f["y_end"] = y_end
        f["y_block"] = y_block
        f["y"] = y
        f["lat_lon_pid"] = lat_lon_pid

        print(f.tree())
    print(f"Data Written for {fname}")


11122
11337
/
 ├── lat_lon_pid (11122, 3) float64
 ├── x_s (11122, 2) float64
 ├── x_t (11122, 4, 0) float64
 ├── y (11122, 100, 5) float64
 ├── y_block (11122, 4, 5) float64
 ├── y_end (11122, 5) float64
 └── y_start (11122, 5) float64
Data Written for ssp585_2015_2040
11173
11502
/
 ├── lat_lon_pid (11173, 3) float64
 ├── x_s (11173, 2) float64
 ├── x_t (11173, 4, 0) float64
 ├── y (11173, 100, 5) float64
 ├── y_block (11173, 4, 5) float64
 ├── y_end (11173, 5) float64
 └── y_start (11173, 5) float64
Data Written for ssp126_2015_2040
11269
11417
/
 ├── lat_lon_pid (11269, 3) float64
 ├── x_s (11269, 2) float64
 ├── x_t (11269, 4, 0) float64
 ├── y (11269, 100, 5) float64
 ├── y_block (11269, 4, 5) float64
 ├── y_end (11269, 5) float64
 └── y_start (11269, 5) float64
Data Written for picontrol_2015_2040
11049
11281
/
 ├── lat_lon_pid (11049, 3) float64
 ├── x_s (11049, 2) float64
 ├── x_t (11049, 4, 0) float64
 ├── y (11049, 100, 5) float64
 ├── y_block (11049, 4, 5) float64
 ├── y_en

### Generate data for model 2 (Climate)

In [3]:
for fname in ["ssp585_2015_2040", "ssp126_2015_2040", "picontrol_2015_2040",
              "ssp585_2075_2100","ssp126_2075_2100", "picontrol_2075_2100"]: #add path of data
    ds_path = f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/data_{fname}.csv"

    data = ReadMLData_m2(ds_path)
    x_t, x_s, y_start, y_end, y_block, y, lat_lon_pid = data.generate_ml_data(block_size=25)
    index = ~np.isnan(y_end.sum(axis=1))
    print(sum(index))
    print(y_end.shape[0])

    x_t = x_t[index]
    x_s = x_s[index]
    y_start = y_start[index]
    y_end = y_end[index]
    y_block = y_block[index] 
    y = y[index]
    lat_lon_pid = lat_lon_pid[index]

    
    with zarr.open(f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/{fname}_m2.zarr", "w") as f: #correct path
    
        f["x_t"] = x_t
        f["x_s"] = x_s
        f["y_start"] = y_start
        f["y_end"] = y_end
        f["y_block"] = y_block
        f["y"] = y
        f["lat_lon_pid"] = lat_lon_pid

        print(f.tree())
    print(f"Data Written for {fname}")


11122
11337
/
 ├── lat_lon_pid (11122, 3) float64
 ├── x_s (11122, 5) float64
 ├── x_t (11122, 4, 0) float64
 ├── y (11122, 100, 5) float64
 ├── y_block (11122, 4, 5) float64
 ├── y_end (11122, 5) float64
 └── y_start (11122, 5) float64
Data Written for ssp585_2015_2040
11173
11502
/
 ├── lat_lon_pid (11173, 3) float64
 ├── x_s (11173, 5) float64
 ├── x_t (11173, 4, 0) float64
 ├── y (11173, 100, 5) float64
 ├── y_block (11173, 4, 5) float64
 ├── y_end (11173, 5) float64
 └── y_start (11173, 5) float64
Data Written for ssp126_2015_2040
11269
11417
/
 ├── lat_lon_pid (11269, 3) float64
 ├── x_s (11269, 5) float64
 ├── x_t (11269, 4, 0) float64
 ├── y (11269, 100, 5) float64
 ├── y_block (11269, 4, 5) float64
 ├── y_end (11269, 5) float64
 └── y_start (11269, 5) float64
Data Written for picontrol_2015_2040
11049
11281
/
 ├── lat_lon_pid (11049, 3) float64
 ├── x_s (11049, 5) float64
 ├── x_t (11049, 4, 0) float64
 ├── y (11049, 100, 5) float64
 ├── y_block (11049, 4, 5) float64
 ├── y_en

### Generate data for model 3 (Soil)

In [4]:
for fname in ["ssp585_2015_2040", "ssp126_2015_2040", "picontrol_2015_2040",
              "ssp585_2075_2100","ssp126_2075_2100", "picontrol_2075_2100"]: #add path of data
    ds_path = f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/data_{fname}.csv"

    data = ReadMLData_m3(ds_path)
    x_t, x_s, y_start, y_end, y_block, y, lat_lon_pid = data.generate_ml_data(block_size=25)
    index = ~np.isnan(y_end.sum(axis=1))
    print(sum(index))
    print(y_end.shape[0])

    x_t = x_t[index]
    x_s = x_s[index]
    y_start = y_start[index]
    y_end = y_end[index]
    y_block = y_block[index] 
    y = y[index]
    lat_lon_pid = lat_lon_pid[index]

    
    with zarr.open(f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/{fname}_m3.zarr", "w") as f: #correct path
    
        f["x_t"] = x_t
        f["x_s"] = x_s
        f["y_start"] = y_start
        f["y_end"] = y_end
        f["y_block"] = y_block
        f["y"] = y
        f["lat_lon_pid"] = lat_lon_pid

        print(f.tree())
    print(f"Data Written for {fname}")


11122
11337
/
 ├── lat_lon_pid (11122, 3) float64
 ├── x_s (11122, 6) float64
 ├── x_t (11122, 4, 0) float64
 ├── y (11122, 100, 5) float64
 ├── y_block (11122, 4, 5) float64
 ├── y_end (11122, 5) float64
 └── y_start (11122, 5) float64
Data Written for ssp585_2015_2040
11173
11502
/
 ├── lat_lon_pid (11173, 3) float64
 ├── x_s (11173, 6) float64
 ├── x_t (11173, 4, 0) float64
 ├── y (11173, 100, 5) float64
 ├── y_block (11173, 4, 5) float64
 ├── y_end (11173, 5) float64
 └── y_start (11173, 5) float64
Data Written for ssp126_2015_2040
11269
11417
/
 ├── lat_lon_pid (11269, 3) float64
 ├── x_s (11269, 6) float64
 ├── x_t (11269, 4, 0) float64
 ├── y (11269, 100, 5) float64
 ├── y_block (11269, 4, 5) float64
 ├── y_end (11269, 5) float64
 └── y_start (11269, 5) float64
Data Written for picontrol_2015_2040
11049
11281
/
 ├── lat_lon_pid (11049, 3) float64
 ├── x_s (11049, 6) float64
 ├── x_t (11049, 4, 0) float64
 ├── y (11049, 100, 5) float64
 ├── y_block (11049, 4, 5) float64
 ├── y_en

### Generate data for model 4 (all covariates)

In [2]:
for fname in ["ssp585_2015_2040", "ssp126_2015_2040", "picontrol_2015_2040",
              "ssp585_2075_2100","ssp126_2075_2100", "picontrol_2075_2100"]: #add path of data
    ds_path = f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/data_{fname}.csv"

    data = ReadMLData_m4(ds_path)
    x_t, x_s, y_start, y_end, y_block, y, lat_lon_pid = data.generate_ml_data(block_size=25)
    index = ~np.isnan(y_end.sum(axis=1))
    print(sum(index))
    print(y_end.shape[0])

    x_t = x_t[index]
    x_s = x_s[index]
    y_start = y_start[index]
    y_end = y_end[index]
    y_block = y_block[index] 
    y = y[index]
    lat_lon_pid = lat_lon_pid[index]

    
    with zarr.open(f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/{fname}_m4.zarr", "w") as f: #correct path
    
        f["x_t"] = x_t
        f["x_s"] = x_s
        f["y_start"] = y_start
        f["y_end"] = y_end
        f["y_block"] = y_block
        f["y"] = y
        f["lat_lon_pid"] = lat_lon_pid

        print(f.tree())
    print(f"Data Written for {fname}")

11122
11337
/
 ├── lat_lon_pid (11122, 3) float64
 ├── x_s (11122, 13) float64
 ├── x_t (11122, 4, 0) float64
 ├── y (11122, 100, 5) float64
 ├── y_block (11122, 4, 5) float64
 ├── y_end (11122, 5) float64
 └── y_start (11122, 5) float64
Data Written for ssp585_2015_2040
11173
11502
/
 ├── lat_lon_pid (11173, 3) float64
 ├── x_s (11173, 13) float64
 ├── x_t (11173, 4, 0) float64
 ├── y (11173, 100, 5) float64
 ├── y_block (11173, 4, 5) float64
 ├── y_end (11173, 5) float64
 └── y_start (11173, 5) float64
Data Written for ssp126_2015_2040
11269
11417
/
 ├── lat_lon_pid (11269, 3) float64
 ├── x_s (11269, 13) float64
 ├── x_t (11269, 4, 0) float64
 ├── y (11269, 100, 5) float64
 ├── y_block (11269, 4, 5) float64
 ├── y_end (11269, 5) float64
 └── y_start (11269, 5) float64
Data Written for picontrol_2015_2040
11049
11281
/
 ├── lat_lon_pid (11049, 3) float64
 ├── x_s (11049, 13) float64
 ├── x_t (11049, 4, 0) float64
 ├── y (11049, 100, 5) float64
 ├── y_block (11049, 4, 5) float64
 ├── 

In [2]:
for fname in ["ssp585_2015_2040", "ssp126_2015_2040", "picontrol_2015_2040",
              "ssp585_2075_2100","ssp126_2075_2100", "picontrol_2075_2100"]: #add path of data
    ds_path = f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/data_{fname}.csv"

    data = ReadMLData_m5(ds_path)
    x_t, x_s, y_start, y_end, y_block, y, lat_lon_pid = data.generate_ml_data(block_size=33)
    index = ~np.isnan(y_end.sum(axis=1))
    print(sum(index))
    print(y_end.shape[0])

    x_t = x_t[index]
    x_s = x_s[index]
    y_start = y_start[index]
    y_end = y_end[index]
    y_block = y_block[index] 
    y = y[index]
    lat_lon_pid = lat_lon_pid[index]

    
    with zarr.open(f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/{fname}_m5.zarr", "w") as f: #correct path
    
        f["x_t"] = x_t
        f["x_s"] = x_s
        f["y_start"] = y_start
        f["y_end"] = y_end
        f["y_block"] = y_block
        f["y"] = y
        f["lat_lon_pid"] = lat_lon_pid

        print(f.tree())
    print(f"Data Written for {fname}")

11122
11337
/
 ├── lat_lon_pid (11122, 3) float64
 ├── x_s (11122, 0) float64
 ├── x_t (11122, 3, 5) float64
 ├── y (11122, 100, 5) float64
 ├── y_block (11122, 3, 5) float64
 ├── y_end (11122, 5) float64
 └── y_start (11122, 5) float64
Data Written for ssp585_2015_2040
11173
11502
/
 ├── lat_lon_pid (11173, 3) float64
 ├── x_s (11173, 0) float64
 ├── x_t (11173, 3, 5) float64
 ├── y (11173, 100, 5) float64
 ├── y_block (11173, 3, 5) float64
 ├── y_end (11173, 5) float64
 └── y_start (11173, 5) float64
Data Written for ssp126_2015_2040
11269
11417
/
 ├── lat_lon_pid (11269, 3) float64
 ├── x_s (11269, 0) float64
 ├── x_t (11269, 3, 5) float64
 ├── y (11269, 100, 5) float64
 ├── y_block (11269, 3, 5) float64
 ├── y_end (11269, 5) float64
 └── y_start (11269, 5) float64
Data Written for picontrol_2015_2040
11049
11281
/
 ├── lat_lon_pid (11049, 3) float64
 ├── x_s (11049, 0) float64
 ├── x_t (11049, 3, 5) float64
 ├── y (11049, 100, 5) float64
 ├── y_block (11049, 3, 5) float64
 ├── y_en

In [ ]:
for fname in ["ssp585_2015_2040", "ssp126_2015_2040", "picontrol_2015_2040",
              "ssp585_2075_2100","ssp126_2075_2100", "picontrol_2075_2100"]: #add path of data
    ds_path = f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/data_{fname}.csv"

    data = ReadMLData_m6(ds_path)
    x_t, x_s, y_start, y_end, y_block, y, lat_lon_pid = data.generate_ml_data(block_size=30)
    index = ~np.isnan(y_end.sum(axis=1))
    print(sum(index))
    print(y_end.shape[0])

    x_t = x_t[index]
    x_s = x_s[index]
    y_start = y_start[index]
    y_end = y_end[index]
    y_block = y_block[index] 
    y = y[index]
    lat_lon_pid = lat_lon_pid[index]

    
    with zarr.open(f"/dss/dssfs02/lwp-dss-0001/pr48va/pr48va-dss-0000/ge96dul2/patch_analysis_paper/data/random_forest/{fname}_m6.zarr", "w") as f: #correct path
    
        f["x_t"] = x_t
        f["x_s"] = x_s
        f["y_start"] = y_start
        f["y_end"] = y_end
        f["y_block"] = y_block
        f["y"] = y
        f["lat_lon_pid"] = lat_lon_pid

        print(f.tree())
    print(f"Data Written for {fname}")

11122
11337
/
 ├── lat_lon_pid (11122, 3) float64
 ├── x_s (11122, 8) float64
 ├── x_t (11122, 3, 5) float64
 ├── y (11122, 100, 5) float64
 ├── y_block (11122, 3, 5) float64
 ├── y_end (11122, 5) float64
 └── y_start (11122, 5) float64
Data Written for ssp585_2015_2040
11173
11502
/
 ├── lat_lon_pid (11173, 3) float64
 ├── x_s (11173, 8) float64
 ├── x_t (11173, 3, 5) float64
 ├── y (11173, 100, 5) float64
 ├── y_block (11173, 3, 5) float64
 ├── y_end (11173, 5) float64
 └── y_start (11173, 5) float64
Data Written for ssp126_2015_2040
11269
11417
/
 ├── lat_lon_pid (11269, 3) float64
 ├── x_s (11269, 8) float64
 ├── x_t (11269, 3, 5) float64
 ├── y (11269, 100, 5) float64
 ├── y_block (11269, 3, 5) float64
 ├── y_end (11269, 5) float64
 └── y_start (11269, 5) float64
Data Written for picontrol_2015_2040
